# Stage 03: Localised (span-level) PCL detection (Part 5)

Utilises task2 dataset span text to get pure localised signals of where in paragh pcl is actually occuring, so the model can aggreagate these local signals and global context for more accurate paragraph-level predictions

steps: ......

## Imports & Dataset utilities

In [7]:
from pathlib import Path
import pandas as pd
import numpy as np
import sys
import os
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel
from transformers import get_linear_schedule_with_warmup

# project root (one level above notebooks)
ROOT = Path().resolve().parents[0]
sys.path.append(str(ROOT))

from src.data.make_dataset import build_task1_task2_with_spans, validate_span_ranges, validate_span_text_alignment, truncation_rate_by_label
from src.training.metrics import stats_on_loader
from src.training.tokenization_utils import ensure_token_cache, TensorCacheDataset, PCLTokenDataset

RAW_TASK1 = ROOT / "data" / "raw" / "dontpatronizeme_pcl.tsv"
RAW_TASK2 = ROOT / "data" / "raw" / "dontpatronizeme_categories.tsv"
TRAIN_SPLIT = ROOT / "data" / "splits" / "train_semeval_parids-labels.csv"
DEV_SPLIT = ROOT / "data" / "splits" / "dev_semeval_parids-labels.csv"

train_df, dev_df, pcl_df, spans_df_norm = build_task1_task2_with_spans(
    raw_task1_path=RAW_TASK1,
    raw_task2_path=RAW_TASK2,
    train_split_path=TRAIN_SPLIT,
    dev_split_path=DEV_SPLIT,
)

print("train_df:", train_df.shape, "| positives:", int(train_df["label_bin"].sum()))
print("dev_df:", dev_df.shape, "| positives:", int(dev_df["label_bin"].sum()))
print("Example span_ranges:", train_df.loc[train_df["label_bin"].idxmax(), "span_ranges"] if len(train_df) else None)

train_df: (8375, 8) | positives: 794
dev_df: (2094, 8) | positives: 199
Example span_ranges: [(157, 242)]


## Deterministic Span Sampler for Span-First Training

### Overview: 4-Stage Span-First Training Pipeline

This section implements a **deterministic span sampler** that enables a span-first training strategy for PCL detection. Instead of training only on full paragraphs, we:

1. **Extract and sample spans** from paragraphs using linguistic segmentation
2. **Train a span classifier** (DeBERTa/ALBERT) to identify PCL at the span level
3. **Augment positive spans** with paraphrases for better generalization
4. **Aggregate span predictions** to make paragraph-level decisions

---

### Stage 1: Build Span Dataset (This Section)

**Goal**: Create a balanced span-level training dataset

**Positive Examples** (PCL spans):
- Use annotated Task2 spans from `spans_df_norm`
- These are human-labeled patronizing/condescending phrases

**Negative Examples** (non-PCL spans):
- Deterministically sample spans from non-PCL paragraphs (Task1 label=0)
- Match annotated span length distribution using quantile grid
- Ensure coverage with deterministic anchor positions

**Key Features**:
- ✅ **Deterministic**: No randomness, reproducible sampling
- ✅ **Distribution matching**: Samples match annotated span lengths
- ✅ **Sentence-aware**: Individual sentences/clauses captured as spans
- ✅ **Context preservation**: BOTH full sentences AND comma-split clauses included
- ✅ **Coverage**: Anchors + accumulation ensure all regions sampled
- ✅ **Quality filters**: Rejects mid-word starts, incomplete words, single-word spans
- ✅ **Word completion**: Always finishes last word (never cuts mid-word)
- ✅ **Text cleaning**: Removes HTML tags, normalizes entities and whitespace
- ✅ **High diversity**: Lower IoU threshold (0.4) and more anchors (4-15) for variety

---

### Algorithm Details

**1. Text Cleaning** (`clean_text_light`):
   - Remove HTML tags (`<h>`, `<p>`, `<div>`, etc.)
   - Remove HTML entities (`&nbsp;`, `&quot;`, etc.)
   - Normalize whitespace and punctuation spacing
   - Preserve sentence structure and character offsets

**2. Segmentation** (`segment_paragraph_into_units`):
   - Use spaCy for sentence segmentation
   - **Dual-mode segmentation**: Returns BOTH full sentences AND comma-split clauses
   - **Aggressive clause splitting**: Split on `,` `;` `—` even without surrounding spaces
   - Each unit = sentence or clause with char offsets
   - Captures short phrases like "dwindling hope" via comma splits
   - Preserves full sentences with commas for context

**3. Quantile-based Length Selection** (`compute_span_length_quantiles`):
   - Analyze annotated span distribution
   - Extract quantiles: [0.05, 0.10, 0.20, 0.35, 0.50, 0.65, 0.80, 0.90, 0.95]
   - Returns target lengths (in tokens) for sampling

**4. Deterministic Anchors** (`compute_deterministic_anchors`):
   - Space anchors across paragraph: `n_anchors = 4-15` (boosted for short paragraphs)
   - Evenly distribute using `np.linspace`
   - Ensures all regions have chance to contribute spans
   - Boost for short paragraphs: `min(n_units, min_anchors * 2)` for better coverage

**5. Span Construction** (`sample_spans_for_paragraph`):
   
   **Phase 1 - Individual units**: Add each sentence/clause as a span candidate
   - Captures distinct semantic units (single sentences, comma-separated phrases)
   - Ensures short, focused spans are included
   - Filters out single-word spans (uninformative, could cause keyword triggering)
   - Example: "dwindling hope" (split on comma) becomes a span
   
   **Phase 2 - Accumulated spans**: For each (anchor, target_length) pair:
   - Accumulate units forward from anchor until reaching target length
   - Extract text using char offsets
   - Align to word boundaries (avoid mid-word starts)
   - **Complete last word**: Extend span end to finish partial words
   - Apply quality filters (including single-word filter)
   - Example: Combines "They need our help." + "Without aid..." for longer spans
   
   **Phase 3 - Deduplication**:
   - Remove spans with **IoU > 0.4** (lower threshold for higher diversity)
   - Remove substring containment (span A fully inside span B)
   - Keep up to `max_spans_per_par=20` unique spans

**6. Quality Filters** (`is_well_formed_span`):
   - Reject leading punctuation (commas, periods, etc.)
   - Reject mid-word starts (e.g., "n ,", "ase of")
   - Reject incomplete words at start/end (e.g., "dise-")
   - **Reject single-word spans** (e.g., "Unfortunately,") - uninformative, 2+ words required
   - Require minimum length (3+ chars)
   - Require alphanumeric content

**7. Word Boundary Completion**:
   - **Start**: If span starts mid-word, back up to word start
   - **End**: If span ends mid-word, extend to word end
   - Ensures spans like "deserves better treatment" not "serves better tre"

---

### Example Output

**Input paragraph**:
> "Refugees need help. They deserve compassion. Without aid, they perish."

**Generated spans** (mix of individual + accumulated):
- "Refugees need help." (full sentence)
- "They deserve compassion." (full sentence)  
- "Without aid, they perish." (full sentence)
- "Refugees need help. They deserve compassion." (2 sentences accumulated)
- "They deserve compassion. Without aid, they perish." (2 sentences accumulated)
- (Full paragraph span if within target lengths)

**With commas** (e.g., "But despite the dwindling hope, Yemenis refuse..."):
**NEW: Preserves BOTH context and focused phrases:**
- "But despite the dwindling hope, Yemenis refuse to give up on others in need." (full sentence with commas)
- "But despite the dwindling hope," (before comma - focused phrase)
- "Yemenis refuse to give up on others in need." (after comma - focused phrase)
- ~~"Unfortunately,"~~ (filtered - single-word span)
- (Deduplication removes high IoU overlaps)

---

### Downstream Usage

**Stage 2**: Train span classifier on `span_train_df`
**Stage 3**: Augment positive spans with paraphrasing
**Stage 4**: Use `span_dev_df` for evaluation - score all spans, aggregate to paragraph level

In [8]:
# Import span sampler functions from module
import sys
import importlib
sys.path.append('../src')

# Force reload to get latest changes
import data.span_sampler
importlib.reload(data.span_sampler)

from data.span_sampler import (
    word_tokenize,
    TextUnit,
    SpanAnnotation,
    segment_paragraph_into_units,
    compute_span_length_quantiles,
    compute_deterministic_anchors,
    is_well_formed_span,
    compute_span_overlap_iou,
    sample_spans_for_paragraph,
    build_span_training_dataset,
    build_eval_span_candidates,
)

print("✓ Span sampler functions imported from src/data/span_sampler.py")
print("✓ Module reloaded - using latest code")

✓ Span sampler functions imported from src/data/span_sampler.py
✓ Module reloaded - using latest code


In [9]:
# Build the span datasets for training and evaluation (Stage 1)
print("="*80)
print("BUILDING SPAN DATASETS (Stage 1)")
print("="*80)

if 'spans_df_norm' in dir() and spans_df_norm is not None and not spans_df_norm.empty:
    # Compute target lengths from annotated spans
    target_lengths = compute_span_length_quantiles(spans_df_norm)
    median_len = int(spans_df_norm['span_text'].apply(lambda x: len(word_tokenize(x))).median())
    
    print(f"\nSpan length distribution from annotations:")
    print(f"  Quantile grid: {target_lengths}")
    print(f"  Median: {median_len} tokens")
    
    # Build training dataset (positive + negative spans)
    print("\n" + "="*80)
    span_train_df = build_span_training_dataset(
        train_df=train_df,
        spans_df=spans_df_norm,
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
        negative_ratio=1.0,  # Balanced classes
    )
    
    # Build evaluation span candidates
    print("\n" + "="*80)
    span_dev_df = build_eval_span_candidates(
        eval_df=dev_df,
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
    )
    
    print("\n" + "="*80)
    print("✓ Span datasets ready for Stage 2 (Span Classifier Training)")
    print("="*80)
else:
    print("ERROR: spans_df_norm not loaded. Run data loading cell (Cell 5) first.")
    span_train_df = None
    span_dev_df = None

BUILDING SPAN DATASETS (Stage 1)

Span length distribution from annotations:
  Quantile grid: [3, 4, 7, 10, 14, 18, 24, 31, 39]
  Median: 14 tokens



Positive spans: 100%|██████████| 2760/2760 [00:00<00:00, 40138.97it/s]



Sampling 2760 negative spans from non-PCL paragraphs...


Negative spans:   7%|▋         | 540/7581 [00:04<00:53, 132.15it/s]



✓ Span dataset built:
  Positive (PCL): 2,760
  Negative (non-PCL): 2,760
  Total: 5,520
  Class balance: {1: 0.5, 0: 0.5}

Generating eval span candidates for 2094 paragraphs...


Eval spans: 100%|██████████| 2094/2094 [00:11<00:00, 176.33it/s]


✓ Eval span candidates built:
  Total spans: 10,562
  Avg spans per paragraph: 5.0
  Paragraphs: PCL=199, non-PCL=1895

✓ Span datasets ready for Stage 2 (Span Classifier Training)


In [10]:
# Quick test: span sampler on example paragraph
test_paragraph = """Refugees are among the most vulnerable populations in the world. They desperately need our help and compassion. Without international aid, they would surely perish. We must do everything we can to save them from their terrible fate."""

print("="*80)
print("QUICK TEST: Span Sampler on Example Paragraph")
print("="*80)

if 'spans_df_norm' in dir() and spans_df_norm is not None and not spans_df_norm.empty:
    target_lengths = compute_span_length_quantiles(spans_df_norm)
    median_len = int(spans_df_norm['span_text'].apply(lambda x: len(word_tokenize(x))).median())
    
    print(f"\nTarget lengths: {target_lengths}")
    print(f"Median: {median_len} tokens")
    print(f"\nParagraph ({len(test_paragraph)} chars):\n{test_paragraph}\n")
    
    test_spans = sample_spans_for_paragraph(
        text=test_paragraph.strip(),
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20
    )
    
    print(f"{'='*80}")
    print(f"Generated {len(test_spans)} spans:\n")
    for i, span in enumerate(test_spans, 1):
        # Show FULL text (NO truncation) for quality inspection
        print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
        print()
else:
    print("  Skipped - run data loading cell first")

QUICK TEST: Span Sampler on Example Paragraph

Target lengths: [3, 4, 7, 10, 14, 18, 24, 31, 39]
Median: 14 tokens

Paragraph (232 chars):
Refugees are among the most vulnerable populations in the world. They desperately need our help and compassion. Without international aid, they would surely perish. We must do everything we can to save them from their terrible fate.

Generated 10 spans:

[1] 10 tok | Refugees are among the most vulnerable populations in the world.

[2]  7 tok | They desperately need our help and compassion.

[3]  7 tok | Without international aid, they would surely perish.

[4]  3 tok | Without international aid

[5]  4 tok | they would surely perish.

[6] 13 tok | We must do everything we can to save them from their terrible fate.

[7] 17 tok | Refugees are among the most vulnerable populations in the world. They desperately need our help and compassion.

[8] 24 tok | Refugees are among the most vulnerable populations in the world. They desperately need our help an

In [39]:
import random
import time

# Generate random seed for reproducibility
run_seed = int(time.time() * 1000) % 100000
random.seed(run_seed)
np.random.seed(run_seed)

print("="*80)
print("COMPREHENSIVE SPAN SAMPLING TEST")
print("="*80)
print(f"\n🎲 Random seed for this run: {run_seed}")
print("   (Use this seed to replicate results)\n")

# Test 1: Non-PCL paragraph
non_pcl = train_df[train_df['label_bin'] == 0].sample(1, random_state=run_seed).iloc[0]
print("\n[TEST 1] NON-PCL PARAGRAPH")
print(f"par_id: {non_pcl['par_id']}")
print(f"\nFull text ({len(non_pcl['text'])} chars):")
print(non_pcl['text'])

non_pcl_spans = sample_spans_for_paragraph(
    text=non_pcl['text'],
    target_lengths=target_lengths,
    median_span_len=median_len,
    max_spans_per_par=20,
)

print(f"\n{'='*80}")
print(f"Generated {len(non_pcl_spans)} spans from non-PCL paragraph:")
if len(non_pcl_spans) <= 8:
    # Show all if 8 or fewer
    display_spans = non_pcl_spans
    print("(showing all)")
else:
    # Show random sample if more than 8
    display_spans = random.sample(non_pcl_spans, 8)
    print("(showing random 8)")

print()
for i, span in enumerate(display_spans, 1):
    print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
    print()

# Test 2: PCL paragraph with ground truth
pcl_with_spans = train_df[train_df['label_bin'] == 1].sample(1, random_state=run_seed + 1).iloc[0]
print("="*80)
print("[TEST 2] PCL PARAGRAPH WITH GROUND TRUTH")
print(f"par_id: {pcl_with_spans['par_id']}")
print(f"\nFull text ({len(pcl_with_spans['text'])} chars):")
print(pcl_with_spans['text'])

gt_spans = spans_df_norm[spans_df_norm['par_id'] == pcl_with_spans['par_id']]
print(f"\n{'='*80}")
print(f"Ground truth PCL spans ({len(gt_spans)}):\n")
for i, (_, row) in enumerate(gt_spans.iterrows(), start=1):
    print(f"[{i}] {len(word_tokenize(row['span_text'])):2d} tok | {row['span_text']}")
    print()

pcl_sampled_spans = sample_spans_for_paragraph(
    text=pcl_with_spans['text'],
    target_lengths=target_lengths,
    median_span_len=median_len,
    max_spans_per_par=20,
)

print(f"{'='*80}")
print(f"Sampled spans from PCL paragraph ({len(pcl_sampled_spans)}):")
if len(pcl_sampled_spans) <= 8:
    display_spans = pcl_sampled_spans
    print("(showing all)\n")
else:
    display_spans = random.sample(pcl_sampled_spans, 8)
    print("(showing random 8)\n")

for i, span in enumerate(display_spans, 1):
    print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
    print()

# Test 3: Overlap analysis
print("="*80)
print("[TEST 3] OVERLAP ANALYSIS: Sampled vs Ground Truth")
print("="*80)

for gt_idx, (_, gt_row) in enumerate(gt_spans.iterrows(), start=1):
    gt_start = gt_row['span_start_norm']
    gt_end = gt_row['span_finish_norm']
    
    best_iou = 0.0
    best_match = None
    
    for sampled in pcl_sampled_spans:
        iou = compute_span_overlap_iou(
            (gt_start, gt_end),
            (sampled['span_start_char'], sampled['span_end_char'])
        )
        if iou > best_iou:
            best_iou = iou
            best_match = sampled
    
    print(f"\nGT span {gt_idx}:")
    print(f"  {gt_row['span_text']}")
    if best_match and best_iou > 0.1:
        print(f"  ✓ Best match (IoU={best_iou:.2f}):")
        print(f"    {best_match['span_text']}")
    else:
        print(f"  ✗ No close match (best IoU={best_iou:.2f})")

COMPREHENSIVE SPAN SAMPLING TEST

🎲 Random seed for this run: 43886
   (Use this seed to replicate results)


[TEST 1] NON-PCL PARAGRAPH
par_id: 6379

Full text (151 chars):
The scenes of the accident looked gruesome , with the car toppled over on the road , but fortunately the two women in the car suffered minor injuries .

Generated 5 spans from non-PCL paragraph:
(showing all)

[1] 26 tok | The scenes of the accident looked gruesome, with the car toppled over on the road, but fortunately the two women in the car suffered minor injuries.

[2]  7 tok | The scenes of the accident looked gruesome

[3]  8 tok | with the car toppled over on the road

[4] 11 tok | but fortunately the two women in the car suffered minor injuries.

[5] 16 tok | The scenes of the accident looked gruesome , with the car toppled over on the road

[TEST 2] PCL PARAGRAPH WITH GROUND TRUTH
par_id: 9309

Full text (176 chars):
" Another recent accomplishment is the return in an orderly , dignified and safe manner o

In [12]:
# STATISTICAL COMPARISON: PCL vs Non-PCL Sampled Spans vs ACTUAL POSITIVE SPANS

print("="*80)
print("STATISTICAL ANALYSIS: Span Sampling Performance")
print("="*80)

# Generate random seed for this run (for reproducibility)
import time
run_seed = int(time.time() * 1000) % 100000  # Use timestamp for true randomness
random.seed(run_seed)
np.random.seed(run_seed)

print(f"\n🎲 Random seed for this run: {run_seed}")
print("   (Use this seed to replicate results)")

# Sample from multiple paragraphs
n_test = 20

pcl_pars = train_df[train_df['label_bin'] == 1].sample(n_test, random_state=run_seed)
non_pcl_pars = train_df[train_df['label_bin'] == 0].sample(n_test, random_state=run_seed + 1)

pcl_all_spans = []
non_pcl_all_spans = []

print(f"\nSampling from {n_test} PCL and {n_test} non-PCL paragraphs...")

for _, row in pcl_pars.iterrows():
    spans = sample_spans_for_paragraph(
        text=row['text'],
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
    )
    pcl_all_spans.extend(spans)

for _, row in non_pcl_pars.iterrows():
    spans = sample_spans_for_paragraph(
        text=row['text'],
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
    )
    non_pcl_all_spans.extend(spans)

print(f"\n{'='*80}")
print("RESULTS:")
print(f"  PCL:     {n_test} pars → {len(pcl_all_spans)} spans ({len(pcl_all_spans)/n_test:.1f} avg/par)")
print(f"  Non-PCL: {n_test} pars → {len(non_pcl_all_spans)} spans ({len(non_pcl_all_spans)/n_test:.1f} avg/par)")

pcl_lengths = [s['span_token_len'] for s in pcl_all_spans]
non_pcl_lengths = [s['span_token_len'] for s in non_pcl_all_spans]

# Get actual positive span lengths from Task 2 annotations
actual_positive_lengths = spans_df_norm['span_text'].apply(lambda x: len(word_tokenize(x))).tolist()

print(f"\n{'='*80}")
print("SPAN LENGTH DISTRIBUTION COMPARISON:")
print(f"\nActual Positive (Task 2 annotations):")
print(f"  Total: {len(actual_positive_lengths)} spans")
print(f"  Mean={np.mean(actual_positive_lengths):.1f}, Median={np.median(actual_positive_lengths):.0f}, Std={np.std(actual_positive_lengths):.1f}")
print(f"  Min={np.min(actual_positive_lengths)}, Max={np.max(actual_positive_lengths)}")

print(f"\nPCL Sampled (from sampler):")
print(f"  Mean={np.mean(pcl_lengths):.1f}, Median={np.median(pcl_lengths):.0f}, Std={np.std(pcl_lengths):.1f}")
print(f"  Min={np.min(pcl_lengths) if pcl_lengths else 0}, Max={np.max(pcl_lengths) if pcl_lengths else 0}")

print(f"\nNon-PCL Sampled (from sampler):")
print(f"  Mean={np.mean(non_pcl_lengths):.1f}, Median={np.median(non_pcl_lengths):.0f}, Std={np.std(non_pcl_lengths):.1f}")
print(f"  Min={np.min(non_pcl_lengths) if non_pcl_lengths else 0}, Max={np.max(non_pcl_lengths) if non_pcl_lengths else 0}")

# Quality metrics
print(f"\n{'='*80}")
print("QUALITY METRICS:")
pcl_unique_starts = len(set(s['span_start_char'] for s in pcl_all_spans))
non_pcl_unique_starts = len(set(s['span_start_char'] for s in non_pcl_all_spans))
print(f"  Unique span starts: PCL={pcl_unique_starts}/{len(pcl_all_spans)}, Non-PCL={non_pcl_unique_starts}/{len(non_pcl_all_spans)}")
print(f"  Diversity: PCL={pcl_unique_starts/len(pcl_all_spans) if pcl_all_spans else 0:.2%}, Non-PCL={non_pcl_unique_starts/len(non_pcl_all_spans) if non_pcl_all_spans else 0:.2%}")

# Single-word span check (should be 0 after filtering)
pcl_single_word = sum(1 for s in pcl_all_spans if s['span_token_len'] < 2)
non_pcl_single_word = sum(1 for s in non_pcl_all_spans if s['span_token_len'] < 2)
print(f"  Single-word spans: PCL={pcl_single_word}, Non-PCL={non_pcl_single_word} (should be 0)")

# Helper function to clean punctuation spacing for display
def clean_punct_for_display(text):
    """Remove spaces before punctuation marks for cleaner display."""
    text = re.sub(r'\s+([,;.!?])', r'\1', text)  # Remove space before punct
    text = re.sub(r'\s+', ' ', text)  # Normalize multiple spaces
    return text.strip()

import re

# Show random examples (FULL TEXT - NO TRUNCATION)
print(f"\n{'='*80}")
print("RANDOM SAMPLE SPANS (FULL TEXT, CLEANED PUNCTUATION):")
print(f"\nActual Positive (Task 2) examples (5 random):\n")
actual_sample = spans_df_norm.sample(min(5, len(spans_df_norm)), random_state=run_seed)
for i, (_, row) in enumerate(actual_sample.iterrows(), 1):
    clean_text = clean_punct_for_display(row['span_text'])
    tok_len = len(word_tokenize(row['span_text']))
    print(f"[{i}] {tok_len:2d} tok | {clean_text}")
    print()

print("="*80)
print(f"PCL Sampled examples (5 random, seed={run_seed}):\n")
for i, span in enumerate(random.sample(pcl_all_spans, min(5, len(pcl_all_spans))), 1):
    # Clean punctuation spacing for display
    clean_text = clean_punct_for_display(span['span_text'])
    print(f"[{i}] {span['span_token_len']:2d} tok | {clean_text}")
    print()

print("="*80)
print(f"Non-PCL Sampled examples (5 random, seed={run_seed}):\n")
for i, span in enumerate(random.sample(non_pcl_all_spans, min(5, len(non_pcl_all_spans))), 1):
    # Clean punctuation spacing for display
    clean_text = clean_punct_for_display(span['span_text'])
    print(f"[{i}] {span['span_token_len']:2d} tok | {clean_text}")
    print()

STATISTICAL ANALYSIS: Span Sampling Performance

🎲 Random seed for this run: 75372
   (Use this seed to replicate results)

Sampling from 20 PCL and 20 non-PCL paragraphs...

RESULTS:
  PCL:     20 pars → 124 spans (6.2 avg/par)
  Non-PCL: 20 pars → 109 spans (5.5 avg/par)

SPAN LENGTH DISTRIBUTION COMPARISON:

Actual Positive (Task 2 annotations):
  Total: 2760 spans
  Mean=16.3, Median=14, Std=11.7
  Min=1, Max=138

PCL Sampled (from sampler):
  Mean=16.8, Median=13, Std=12.6
  Min=2, Max=64

Non-PCL Sampled (from sampler):
  Mean=15.2, Median=12, Std=11.5
  Min=2, Max=54

QUALITY METRICS:
  Unique span starts: PCL=63/124, Non-PCL=52/109
  Diversity: PCL=50.81%, Non-PCL=47.71%
  Single-word spans: PCL=0, Non-PCL=0 (should be 0)

RANDOM SAMPLE SPANS (FULL TEXT, CLEANED PUNCTUATION):

Actual Positive (Task 2) examples (5 random):

[1] 10 tok | In order to protect the poor and vulnerable amongst us

[2]  5 tok | put food on their tables

[3]  9 tok | the surrounding air of defeat, dejec

## Model training & Param search

In [13]:
# ---- choose backbone here ----
MODEL_CANDIDATES = {
    "deberta": "microsoft/deberta-v3-base",
    "albert": "albert-base-v2",
    "albert_large": "albert-large-v2",
}

MODEL_KEY = os.getenv("PCL_BACKBONE", "albert_large")  # set to "albert" to try ALBERT
MODEL_NAME = MODEL_CANDIDATES[MODEL_KEY]
MAX_LEN = 192

def load_tokenizer(model_name: str):
    # Online first, then local cache fallback (for DNS/no-internet issues)
    try:
        return AutoTokenizer.from_pretrained(model_name, use_fast=True)
    except Exception as e:
        print(f"Tokenizer load failed ({type(e).__name__}: {e}). Trying local cache...")
        return AutoTokenizer.from_pretrained(model_name, use_fast=True, local_files_only=True)

tokenizer = load_tokenizer(MODEL_NAME)



In [14]:
# How to pool token-level logits to single paragraph level logit (used in TokenCLSModel)

class LogitPooler(nn.Module):
    """
    Pool token-level logits (B,T) -> (B,1) using a boolean mask (B,T).

    Modes:
      - "max": masked max over tokens
      - "topk_mean": mean of top-k masked logits (k set by top_k)
    """
    def __init__(self, mode: str = "max", top_k: int = 3, sentinel: float = -1e4):
        super().__init__()
        self.mode = str(mode)
        self.top_k = int(top_k)
        self.sentinel = float(sentinel)

        if self.mode not in {"max", "topk_mean"}:
            raise ValueError(f"Unknown pooling mode: {self.mode}")

    def forward(self, token_logits: torch.Tensor, token_mask: torch.Tensor) -> torch.Tensor:
        """
        token_logits: (B,T) float
        token_mask:   (B,T) bool (True = keep / real tokens)
        returns:      (B,1)
        """
        token_mask = token_mask.to(dtype=torch.bool)

        # Edge-case guard: if a sample has no real tokens, return 0.0 (neutral feature)
        no_real = ~token_mask.any(dim=1, keepdim=True)  # (B,1)

        x = token_logits.masked_fill(~token_mask, self.sentinel)  # (B,T)

        if self.mode == "max":
            pooled = x.max(dim=1, keepdim=True).values  # (B,1)
            pooled = pooled.masked_fill(no_real, 0.0)
            return pooled

        # self.mode == "topk_mean"
        B, T = x.shape
        k = min(self.top_k, T)
        topk_vals = torch.topk(x, k=k, dim=1).values  # (B,k)

        # If k > #real tokens, topk will include sentinel values; exclude them from the mean.
        valid = topk_vals > (self.sentinel + 1.0)
        denom = valid.sum(dim=1, keepdim=True).clamp(min=1)
        pooled = (topk_vals.masked_fill(~valid, 0.0).sum(dim=1, keepdim=True)) / denom
        pooled = pooled.masked_fill(no_real, 0.0)
        return pooled

In [15]:
class TokenCLSModel(nn.Module):
    def __init__(self, model_name, lambda_token=0.3, pool_mode="max", top_k=3, alpha_neg=0.25):
        super().__init__()

        # Online first, then local cache fallback
        try:
            self.encoder = AutoModel.from_pretrained(model_name)
        except Exception as e:
            print(f"Model load failed ({type(e).__name__}: {e}). Trying local cache...")
            self.encoder = AutoModel.from_pretrained(model_name, local_files_only=True)

        hidden = self.encoder.config.hidden_size

        self.token_head = nn.Linear(hidden, 1)
        self.paragraph_head = nn.Linear(hidden + 1, 1)

        self.lambda_token = float(lambda_token)
        self.alpha_neg = float(alpha_neg)

        # modular pooling (swap "max" <-> "topk_mean")
        self.pooler = LogitPooler(mode=pool_mode, top_k=top_k)

    def forward(
        self,
        input_ids,
        attention_mask,
        token_type_ids=None,
        token_labels=None,
        token_loss_mask=None,
        paragraph_label=None,
    ):
        # Some backbones ignore token_type_ids; some accept it.
        # Try passing it; if unsupported, fall back.
        try:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )
        except TypeError:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )

        hidden = outputs.last_hidden_state  # (B,T,H)

        # keep dtype consistent with heads (prevents dtype mismatch issues)
        hidden = hidden.to(dtype=self.token_head.weight.dtype)

        cls = hidden[:, 0]  # (B,H)
        token_logits = self.token_head(hidden).squeeze(-1)  # (B,T)

        # safe token mask for pooling + token-loss
        if token_loss_mask is None:
            token_loss_mask = attention_mask == 1
        token_loss_mask = token_loss_mask.to(dtype=torch.bool)

        pooled = self.pooler(token_logits, token_loss_mask)  # (B,1)

        fused = torch.cat([cls, pooled], dim=1)  # (B,H+1)
        paragraph_logit = self.paragraph_head(fused).squeeze(-1)  # (B,)

        loss_par = None
        if paragraph_label is not None:
            loss_par = F.binary_cross_entropy_with_logits(
                paragraph_logit.float(),
                paragraph_label.float(),
            )

        # Compute token loss independently (enables true token-only warm start)
        loss_tok = None
        if token_labels is not None:
            per_tok = F.binary_cross_entropy_with_logits(
                token_logits.float(),
                token_labels.float(),
                reduction="none",
            )  # (B,T)

            # downweight negatives globally (token_labels==0)
            weights = torch.where(token_labels > 0.5, 1.0, self.alpha_neg).to(per_tok.dtype)  # (B,T)
            per_tok = per_tok * weights

            per_tok = per_tok.masked_fill(~token_loss_mask, 0.0)
            denom = token_loss_mask.sum().clamp(min=1).to(per_tok.dtype)
            loss_tok = per_tok.sum() / denom

        loss = None
        if (loss_par is not None) and (loss_tok is not None):
            loss = loss_par + self.lambda_token * loss_tok
        elif loss_par is not None:
            loss = loss_par
        elif loss_tok is not None:
            loss = self.lambda_token * loss_tok  # for warm start set lambda_token=1.0

        return {
            "loss": loss,
            "loss_par": loss_par,
            "loss_tok": loss_tok,
            "paragraph_logit": paragraph_logit,
            "token_logits": token_logits,
        }

In [16]:
CACHE_DIR = ROOT / "data" / "cache" / f"pcl_tok_{MODEL_KEY}_len{MAX_LEN}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

train_cache = ensure_token_cache(
    train_df,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    cache_path=CACHE_DIR / "train.pt",
    model_key=MODEL_KEY,
    model_name=MODEL_NAME,
)
dev_cache = ensure_token_cache(
    dev_df,
    tokenizer=tokenizer,
    max_len=MAX_LEN,
    cache_path=CACHE_DIR / "dev.pt",
    model_key=MODEL_KEY,
    model_name=MODEL_NAME,
)

train_dataset = TensorCacheDataset(train_cache)
dev_dataset = TensorCacheDataset(dev_cache)

print("train_dataset:", type(train_dataset), "len=", len(train_dataset))
print("dev_dataset:", type(dev_dataset), "len=", len(dev_dataset))

train_dataset: <class 'src.training.tokenization_utils.TensorCacheDataset'> len= 8375
dev_dataset: <class 'src.training.tokenization_utils.TensorCacheDataset'> len= 2094


In [17]:
from torch.optim import AdamW
from tqdm import tqdm
import torch
import numpy as np
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, WeightedRandomSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def _amp_settings(model_key: str, model_name: str):
    key = (model_key or "").lower()
    name = (model_name or "").lower()

    if ("albert" in key) or ("albert" in name):
        enabled = torch.cuda.is_available()
        return enabled, torch.float16, True  # FP16 + GradScaler

    if ("deberta" in key) or ("deberta" in name):
        enabled = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        return enabled, torch.bfloat16, False  # BF16, no GradScaler

    return False, None, False

USE_AMP, AMP_DTYPE, NEEDS_SCALER = _amp_settings(MODEL_KEY, MODEL_NAME)
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

print(
    f"DEVICE={DEVICE} | backbone={MODEL_KEY} | amp={USE_AMP} | amp_dtype={AMP_DTYPE} | "
    f"scaler={bool(scaler.is_enabled())}"
)

model = TokenCLSModel(MODEL_NAME, lambda_token=0.3, pool_mode="max").to(DEVICE)

# train_dataset = PCLTokenDataset(train_df)

# ---- balanced paragraph sampling (approx 50/50 pos/neg) ----
labels = train_df["label_bin"].astype(int).to_numpy()
pos = int(labels.sum())
neg = int(len(labels) - pos)

w_pos = 1.0 / max(pos, 1)
w_neg = 1.0 / max(neg, 1)

sample_weights = torch.tensor([w_pos if y == 1 else w_neg for y in labels], dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),  # keep "epoch" size comparable to dataset size
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    sampler=sampler,   # <- sampler replaces shuffle
)

# separate loaders for F1 computation (no shuffle)
train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
# dev_dataset = PCLTokenDataset(dev_df)
dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

optimizer = AdamW(
    model.parameters(),
    lr=7e-6,
    weight_decay=1e-2,
)

THRESH = 0.5
GRAD_ACCUM = 2  # <- match your optuna best configs; effective batch = 16 * 2 = 32
EPOCHS = 7

total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
warmup_steps = int(0.07 * total_update_steps)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_update_steps,
)



DEVICE=cuda | backbone=albert_large | amp=True | amp_dtype=torch.float16 | scaler=True


/tmp/ipykernel_695301/1639145740.py:25: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))
Loading weights: 100%|██████████| 25/25 [00:00<00:00, 590.42it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [18]:
b = next(iter(train_loader))
print("batch pos rate:", float(b["paragraph_label"].mean()))

batch pos rate: 0.5625


In [19]:
# --- Optuna: setup (NEW CELL) ---
import optuna
from optuna.pruners import MedianPruner
import random, gc

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

OPTUNA_DIR = ROOT / "runs" / "optuna_stage04_localised_tokencls"
OPTUNA_DIR.mkdir(parents=True, exist_ok=True)

storage_url = f"sqlite:///{(OPTUNA_DIR / 'study.db').as_posix()}"
study = optuna.create_study(
    study_name="stage04_localised_tokencls",
    direction="maximize",
    storage=storage_url,
    load_if_exists=True,
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=1, interval_steps=1),
)

print("storage:", storage_url)
print("trials so far:", len(study.trials))

[I 2026-03-03 04:51:18,238] Using an existing study with name 'stage04_localised_tokencls' instead of creating a new one.


storage: sqlite:////home/joshua_killa/doc/y3/PCL-detection/runs/optuna_stage04_localised_tokencls/study.db
trials so far: 21


In [20]:
# --- Optuna: objective (NEW CELL) ---
from optuna import trial
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.optim import AdamW

# fixed knobs (match your notebook)
EPOCHS = 7
BATCH_SIZE = 16
GRAD_ACCUM = 2
THRESH = 0.5
TOKEN_ONLY_EPOCHS = 1
EARLY_STOP_PATIENCE = 4

# reuse the same balancing weights you already computed from train_df
_labels = train_df["label_bin"].astype(int).to_numpy()
_pos = int(_labels.sum())
_neg = int(len(_labels) - _pos)
_w_pos = 1.0 / max(_pos, 1)
_w_neg = 1.0 / max(_neg, 1)
_sample_weights = torch.tensor([_w_pos if y == 1 else _w_neg for y in _labels], dtype=torch.double)

def objective(trial: optuna.Trial):
    set_seed(SEED)

    # requested search params
    alpha_neg = trial.suggest_float("alpha_neg", 0.1, 0.7)
    lambda_token = trial.suggest_float("lambda_token", 0.05, 1.0)
    token_only_epochs = trial.suggest_int("token_only_epochs", 0, 3)

    # extra (useful) params
    lr = trial.suggest_float("lr", 2e-6, 3e-5, log=True)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.1)

    trial.set_user_attr("model_name", MODEL_NAME)
    trial.set_user_attr("model_key", MODEL_KEY)
    trial.set_user_attr("max_len", int(MAX_LEN))
    trial.set_user_attr("amp", bool(USE_AMP))
    trial.set_user_attr("amp_dtype", str(AMP_DTYPE) if AMP_DTYPE is not None else None)
    trial.set_user_attr("grad_accum", int(GRAD_ACCUM))
    trial.set_user_attr("batch_size", int(BATCH_SIZE))
    trial.set_user_attr("epochs", int(EPOCHS))

    sampler = WeightedRandomSampler(_sample_weights, num_samples=len(_sample_weights), replacement=True)
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
    train_eval_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
    dev_eval_loader = DataLoader(dev_dataset, batch_size=64, shuffle=False)

    model = TokenCLSModel(
        MODEL_NAME,
        lambda_token=lambda_token,
        pool_mode="max",
        alpha_neg=alpha_neg,
    ).to(DEVICE)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_update_steps = (len(train_loader) * EPOCHS + GRAD_ACCUM - 1) // GRAD_ACCUM
    warmup_steps = int(0.07 * total_update_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_update_steps,
    )

    local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))

    best_f1 = -1.0
    no_improve = 0

    try:
        for epoch in range(EPOCHS):
            token_only = epoch < token_only_epochs

            # curriculum
            if token_only:
                for p in model.paragraph_head.parameters():
                    p.requires_grad = False
                model.lambda_token = 1.0
            else:
                for p in model.paragraph_head.parameters():
                    p.requires_grad = True
                model.lambda_token = float(lambda_token)

            model.train()
            optimizer.zero_grad(set_to_none=True)

            last_step = 0
            for step, batch in enumerate(train_loader, start=1):
                last_step = step
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                if token_only:
                    batch["paragraph_label"] = None

                if USE_AMP:
                    with torch.autocast(device_type="cuda", dtype=AMP_DTYPE):
                        out = model(**batch)
                        loss = out["loss"]
                else:
                    out = model(**batch)
                    loss = out["loss"]

                loss = loss / GRAD_ACCUM

                if local_scaler.is_enabled():
                    local_scaler.scale(loss).backward()
                else:
                    loss.backward()

                if (step % GRAD_ACCUM) == 0:
                    if local_scaler.is_enabled():
                        local_scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                    if local_scaler.is_enabled():
                        local_scaler.step(optimizer)
                        local_scaler.update()
                    else:
                        optimizer.step()

                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            # flush remainder
            if last_step and (last_step % GRAD_ACCUM) != 0:
                if local_scaler.is_enabled():
                    local_scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

                if local_scaler.is_enabled():
                    local_scaler.step(optimizer)
                    local_scaler.update()
                else:
                    optimizer.step()

                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            # eval: dev f1 is the Optuna score
            dev_stats = stats_on_loader(
                dev_eval_loader,
                model,
                DEVICE,
                USE_AMP,
                AMP_DTYPE,
                threshold=THRESH,
                compute_token_loss=True,
                compute_token_metrics=True,
                limit_batches=None,
            )
            dev_f1 = float(dev_stats["f1"])

            train_stats = stats_on_loader(
                train_eval_loader,
                model,
                DEVICE,
                USE_AMP,
                AMP_DTYPE,
                threshold=THRESH,
                compute_token_loss=True,
                compute_token_metrics=True,
                limit_batches=20,  # quick estimate like your manual loop
            )

            print(
                f"[trial {trial.number}] Epoch {epoch} | "
                f"train loss={train_stats['loss']:.4f} (par={train_stats['loss_par']:.4f}, tok={train_stats['loss_tok']}) | "
                f"train f1={train_stats['f1']:.4f} acc={train_stats['acc']:.4f} | "
                f"train tok_acc={train_stats.get('tok_acc')} | "
                f"dev loss={dev_stats['loss']:.4f} (par={dev_stats['loss_par']:.4f}, tok={dev_stats['loss_tok']}) | "
                f"dev f1={dev_stats['f1']:.4f} acc={dev_stats['acc']:.4f} | "
                f"dev tok_acc={dev_stats.get('tok_acc')}"
            )

            trial.set_user_attr(f"epoch_{epoch}_train", {k: float(v) for k, v in train_stats.items() if isinstance(v, (int, float, np.floating))})
            trial.set_user_attr(f"epoch_{epoch}_dev",   {k: float(v) for k, v in dev_stats.items() if isinstance(v, (int, float, np.floating))})

            # track best epoch too
            if dev_f1 >= best_f1:
                trial.set_user_attr("best_epoch", int(epoch))
                trial.set_user_attr("best_dev_stats", {k: float(v) for k, v in dev_stats.items() if isinstance(v, (int, float, np.floating))})
                trial.set_user_attr("best_train_stats", {k: float(v) for k, v in train_stats.items() if isinstance(v, (int, float, np.floating))})

            trial.report(dev_f1, step=epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            if dev_f1 > best_f1 + 1e-6:
                best_f1 = dev_f1
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= EARLY_STOP_PATIENCE:
                    break

        return float(best_f1)

    except torch.cuda.OutOfMemoryError:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        raise optuna.TrialPruned()
    finally:
        del model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [21]:
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score
import numpy as np

def evaluate(model, dataset):
    loader = DataLoader(dataset, batch_size=32)
    model.eval()

    all_logits = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            out = model(**batch)
            all_logits.append(out["paragraph_logit"].detach().float().cpu())
            all_labels.append(batch["paragraph_label"].detach().float().cpu())

    logits = torch.cat(all_logits).numpy()
    labels = torch.cat(all_labels).numpy().astype(int)

    probs = 1.0 / (1.0 + np.exp(-logits))

    best_f1 = 0.0
    best_thresh = 0.5
    for t in np.linspace(0.1, 0.9, 81):
        preds = (probs > t).astype(int)
        f1 = f1_score(labels, preds, pos_label=1)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = float(t)

    return best_f1, best_thresh

In [22]:
N_TRIALS = 20
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("best value (dev f1):", study.best_value)
print("best params:", study.best_params)

Loading weights: 100%|██████████| 25/25 [00:00<00:00, 510.16it/s, Materializing param=pooler.weight]                                                             
AlbertModel LOAD REPORT from: albert-large-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
predictions.dense.weight     | UNEXPECTED |  | 
predictions.decoder.bias     | UNEXPECTED |  | 
predictions.bias             | UNEXPECTED |  | 
predictions.LayerNorm.bias   | UNEXPECTED |  | 
predictions.dense.bias       | UNEXPECTED |  | 
predictions.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_695301/1146678584.py:65: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  local_scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and NEEDS_SCALER))
[W 2026-03-03 04:51:51,791] Trial 21 fail

KeyboardInterrupt: 

In [23]:
torch.cuda.empty_cache()

In [ ]:
import random

print("="*80)
print("COMPREHENSIVE SPAN SAMPLING TEST")
print("="*80)

# Test 1: Non-PCL paragraph
non_pcl = train_df[train_df['label_bin'] == 0].sample(1, random_state=42).iloc[0]
print("\n[TEST 1] NON-PCL PARAGRAPH")
print(f"par_id: {non_pcl['par_id']}")
print(f"\nFull text ({len(non_pcl['text'])} chars):")
print(non_pcl['text'])

non_pcl_spans = sample_spans_for_paragraph(
    text=non_pcl['text'],
    target_lengths=target_lengths,
    median_span_len=median_len,
    max_spans_per_par=20,
)

print(f"\n{'='*80}")
print(f"Generated {len(non_pcl_spans)} spans from non-PCL paragraph:")
if len(non_pcl_spans) <= 8:
    # Show all if 8 or fewer
    display_spans = non_pcl_spans
    print("(showing all)")
else:
    # Show random sample if more than 8
    display_spans = random.sample(non_pcl_spans, 8)
    print("(showing random 8)")

print()
for i, span in enumerate(display_spans, 1):
    print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
    print()

# Test 2: PCL paragraph with ground truth
pcl_with_spans = train_df[train_df['label_bin'] == 1].sample(1, random_state=43).iloc[0]
print("="*80)
print("[TEST 2] PCL PARAGRAPH WITH GROUND TRUTH")
print(f"par_id: {pcl_with_spans['par_id']}")
print(f"\nFull text ({len(pcl_with_spans['text'])} chars):")
print(pcl_with_spans['text'])

gt_spans = spans_df_norm[spans_df_norm['par_id'] == pcl_with_spans['par_id']]
print(f"\n{'='*80}")
print(f"Ground truth PCL spans ({len(gt_spans)}):\n")
for i, (_, row) in enumerate(gt_spans.iterrows(), start=1):
    print(f"[{i}] {len(word_tokenize(row['span_text'])):2d} tok | {row['span_text']}")
    print()

pcl_sampled_spans = sample_spans_for_paragraph(
    text=pcl_with_spans['text'],
    target_lengths=target_lengths,
    median_span_len=median_len,
    max_spans_per_par=20,
)

print(f"{'='*80}")
print(f"Sampled spans from PCL paragraph ({len(pcl_sampled_spans)}):")
if len(pcl_sampled_spans) <= 8:
    display_spans = pcl_sampled_spans
    print("(showing all)\n")
else:
    display_spans = random.sample(pcl_sampled_spans, 8)
    print("(showing random 8)\n")

for i, span in enumerate(display_spans, 1):
    print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
    print()

# Test 3: Overlap analysis
print("="*80)
print("[TEST 3] OVERLAP ANALYSIS: Sampled vs Ground Truth")
print("="*80)

for gt_idx, (_, gt_row) in enumerate(gt_spans.iterrows(), start=1):
    gt_start = gt_row['span_start_norm']
    gt_end = gt_row['span_finish_norm']
    
    best_iou = 0.0
    best_match = None
    
    for sampled in pcl_sampled_spans:
        iou = compute_span_overlap_iou(
            (gt_start, gt_end),
            (sampled['span_start_char'], sampled['span_end_char'])
        )
        if iou > best_iou:
            best_iou = iou
            best_match = sampled
    
    print(f"\nGT span {gt_idx}:")
    print(f"  {gt_row['span_text']}")
    if best_match and best_iou > 0.1:
        print(f"  ✓ Best match (IoU={best_iou:.2f}):")
        print(f"    {best_match['span_text']}")
    else:
        print(f"  ✗ No close match (best IoU={best_iou:.2f})")


== train_df ==
Paragraph balance: {'rows': 8375, 'pos': 794, 'neg': 7581, 'pos_rate': 0.09480597014925374, 'pos_weight': 9.547858942065492}
Span coverage (overall): {'rows': 8375, 'total_chars': 2243329, 'span_chars': 145477, 'nonspan_chars': 2097852, 'span_frac': 0.06484871367507843}
Span coverage (positives only): {'rows': 794, 'total_chars': 227501, 'span_chars': 145477, 'nonspan_chars': 82024, 'span_frac': 0.6394565298614072}
Span coverage (negatives only): {'rows': 7581, 'total_chars': 2015828, 'span_chars': 0, 'nonspan_chars': 2015828, 'span_frac': 0.0}
Pos-only per-row span_frac: mean=0.693176 | median=0.739633 | p90=0.987998
Neg-only per-row span_frac: mean=0.000000 | median=0.000000 | p90=0.000000
Approx token pos_weight (overall char nonspan/span): 14.420506334334638
Approx token pos_weight (pos-only char nonspan/span): 0.5638279590588203

== dev_df ==
Paragraph balance: {'rows': 2094, 'pos': 199, 'neg': 1895, 'pos_rate': 0.09503342884431709, 'pos_weight': 9.522613065326633}

In [ ]:
# STATISTICAL COMPARISON: PCL vs Non-PCL Sampled Spans

print("="*80)
print("STATISTICAL ANALYSIS: Span Sampling Performance")
print("="*80)

# Sample from multiple paragraphs
n_test = 20
random.seed(42)

pcl_pars = train_df[train_df['label_bin'] == 1].sample(n_test, random_state=44)
non_pcl_pars = train_df[train_df['label_bin'] == 0].sample(n_test, random_state=45)

pcl_all_spans = []
non_pcl_all_spans = []

print(f"\nSampling from {n_test} PCL and {n_test} non-PCL paragraphs...")

for _, row in pcl_pars.iterrows():
    spans = sample_spans_for_paragraph(
        text=row['text'],
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
    )
    pcl_all_spans.extend(spans)

for _, row in non_pcl_pars.iterrows():
    spans = sample_spans_for_paragraph(
        text=row['text'],
        target_lengths=target_lengths,
        median_span_len=median_len,
        max_spans_per_par=20,
    )
    non_pcl_all_spans.extend(spans)

print(f"\n{'='*80}")
print("RESULTS:")
print(f"  PCL:     {n_test} pars → {len(pcl_all_spans)} spans ({len(pcl_all_spans)/n_test:.1f} avg/par)")
print(f"  Non-PCL: {n_test} pars → {len(non_pcl_all_spans)} spans ({len(non_pcl_all_spans)/n_test:.1f} avg/par)")

pcl_lengths = [s['span_token_len'] for s in pcl_all_spans]
non_pcl_lengths = [s['span_token_len'] for s in non_pcl_all_spans]

print(f"\nSpan Length Statistics:")
print(f"  PCL:     mean={np.mean(pcl_lengths):.1f}, median={np.median(pcl_lengths):.0f}, std={np.std(pcl_lengths):.1f}")
print(f"  Non-PCL: mean={np.mean(non_pcl_lengths):.1f}, median={np.median(non_pcl_lengths):.0f}, std={np.std(non_pcl_lengths):.1f}")

# Quality metrics
print(f"\n{'='*80}")
print("QUALITY METRICS:")
pcl_unique_starts = len(set(s['span_start_char'] for s in pcl_all_spans))
non_pcl_unique_starts = len(set(s['span_start_char'] for s in non_pcl_all_spans))
print(f"  Unique span starts: PCL={pcl_unique_starts}/{len(pcl_all_spans)}, Non-PCL={non_pcl_unique_starts}/{len(non_pcl_all_spans)}")
print(f"  Diversity: PCL={pcl_unique_starts/len(pcl_all_spans) if pcl_all_spans else 0:.2%}, Non-PCL={non_pcl_unique_starts/len(non_pcl_all_spans) if non_pcl_all_spans else 0:.2%}")

# Show random examples (FULL TEXT)
print(f"\n{'='*80}")
print("RANDOM SAMPLE SPANS (FULL TEXT):")
print("\nPCL examples (5 random):\n")
for i, span in enumerate(random.sample(pcl_all_spans, min(5, len(pcl_all_spans))), 1):
    print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
    print()

print("="*80)
print("Non-PCL examples (5 random):\n")
for i, span in enumerate(random.sample(non_pcl_all_spans, min(5, len(non_pcl_all_spans))), 1):
    print(f"[{i}] {span['span_token_len']:2d} tok | {span['span_text']}")
    print()